# Quantifying Biases in LLM-as-a-Judge Evaluations

::::::::::::{div}
:class: literature-paper judge-bias-review

:::{div}
:class: paper-metadata

**Magda Dubois · Harry Coppock · Mario Giulianelli · Ole Kristian Jorgensen · Timo Flesch · Lennart Luettgau · Cozmin Ududec**

ICML 2026 · Proceedings of Machine Learning Research 306, pp. 27120–27135

[Published paper](https://proceedings.mlr.press/v306/dubois26a.html) · [HiBayES GitHub](https://github.com/UKGovernmentBEIS/hibayes) · [Software documentation](https://ukgovernmentbeis.github.io/hibayes/)
:::

The paper separates LLM performance from systematic judge behavior: strictness, preference for its own model, specific to an item disagreement and length preference. **The five methodological examples use simulated data; the final application analyzes real MT Bench judgments.**

## Reading the model notation

The paper writes $\phi_i$ (phi) for the **linear predictor**. This review writes $\eta_i$ (eta) for that same quantity. They are two names for the same model component, not two different effects. The predictor combines the model's terms into a continuous number, which the likelihood then converts into probabilities of the possible ratings or choices.

| Symbol | What it means |
| --- | --- |
| $i$ | One recorded judgment. One answer can receive several judgments from different graders. |
| $g_i$ | The grader who supplied judgment $i$, such as a human or a particular automatic judge. |
| $\ell_i$ | The LLM that generated the answer being judged. |
| $m_i$ | The question associated with judgment $i$. Here $m_i$ identifies a question, not an evaluation metric. |
| $Y_i$ | The actual rating assigned by the grader. In an ordinal model, it must be one of the allowed categories. |
| $x_i$ | A known predictor, such as a numerical code for grader identity or a measured length difference. |
| $\beta_0,\beta_1,\beta_2$ | The baseline and regression coefficients. They determine how the predictor changes with grader and generator identity. |
| $\phi_i$ in the paper; $\eta_i$ here | The linear predictor for judgment $i$. For example, $\eta_i=\beta_0+\beta_1x_i^{\mathrm{grader}}+\beta_2x_i^{\mathrm{LLM}}$. It is a modelled scoring tendency on a continuous latent scale. |
| $c=(c_1,\ldots,c_{K-1})$ | Ordered boundaries for $K$ rating categories. Together with $\eta_i$, they determine the probability of each category. |
| $\theta$ | A shorthand for **all unknown parameters in the model being discussed**. In the first ordinal model, $\theta=(\beta_0,\beta_1,\beta_2,c)$. In a hierarchical model, it also includes the relevant group means, spreads and individual effects. |
| $D$ | The complete observed dataset: judgments, grader and generator identities, questions, and any recorded predictors used by the model. |
| $p(\theta\mid D)$ | The posterior distribution: which parameter combinations remain plausible after observing $D$, under the model and its priors. |
| $\mathcal N(\mu,\sigma^2)$ | A normal distribution with mean $\mu$ and variance $\sigma^2$. Its standard deviation is $\sigma$. This review consistently uses **variance** as the second argument. |

### How the predictor differs from the parameter collection

For one possible parameter set $\theta$, the model computes one $\eta_i$ for each judgment. A different plausible parameter set generally produces a different $\eta_i$. Posterior sampling therefore gives uncertainty in the predictor and in the resulting score probabilities. Neither $\phi_i$ nor $\eta_i$ is a directly measured “true quality” score.

:::{admonition} Worked example: parameters, predictor and rating probability
:class: tip ds-example-card

Suppose one illustrative parameter set has $\beta_0=0.4$, $\beta_1=-0.6$ and $\beta_2=0.8$. For a judgment with both identity codes equal to $+1$,

$$
\eta_i=\phi_i=0.4-0.6(1)+0.8(1)=0.6.
$$

The value 0.6 is **not a rating of 0.6 and not a 60% probability**. If this parameter set also has $c_5=0.6$, the ordinal likelihood gives

$$
P(Y_i\leq5\mid\theta)=\frac{1}{1+\exp[-(c_5-\eta_i)]}=0.5.
$$

So the model assigns a 50% probability to a rating of 5 or below. The full set of cutpoints gives probabilities for every category. Here $\theta$ contains the coefficients **and** those cutpoints; $\eta_i$ is calculated from them and the known identity codes. These numbers illustrate the notation and are not fitted findings from the paper.
:::


## Grader strictness

#### Ordered logistic regression

Include both grader and generating LLM to distinguish a stricter judge from a genuinely lower scoring model. Ordered logistic is appropriate because the 1–10 ratings have a meaningful order, but a one point difference need not represent the same quality change everywhere on the scale.

$$
\eta_i=\beta_0+\beta_1x_i^{\mathrm{grader}}+\beta_2x_i^{\mathrm{LLM}}.
$$

Here $\beta_0$ is the baseline latent scoring level, $\beta_1$ represents the grader effect, and $\beta_2$ represents the generating LLM effect. The $x$ variables are known identity codes, whereas the $\beta$ values are estimated from the ratings. Their combination $\eta_i$ is continuous, but the recorded outcome $Y_i$ must be a score category.

The vector **$c$ contains cutpoints**, the boundaries separating adjacent score categories on that latent scale. For the text's 1–10 scale there are nine boundaries, $c_1<\cdots<c_9$: $c_1$ separates category 1 from categories above it, $c_2$ separates categories up to 2 from those above 2, and so on. These are not the numbers 1, 2, …, 9 themselves, and their spacing need not be equal.

The model connects the predictor to the boundaries through

$$
P(Y_i\le k\mid\beta,c)=\sigma(c_k-\eta_i),
\qquad \sigma(z)=\frac{1}{1+e^{-z}}.
$$

Thus $c_k=\eta_i$ means a 50% probability of a score at or below $k$. A larger $\eta_i$ lowers that cumulative probability and shifts probability toward higher scores. Subtracting adjacent cumulative probabilities gives the probability of one category:

$$
P(Y_i=k\mid\beta,c)
=\sigma(c_k-\eta_i)-\sigma(c_{k-1}-\eta_i).
$$

The end categories use $c_0=-\infty$ and $c_{10}=+\infty$. These probabilities, evaluated at the actual observed scores, supply the likelihood.

:::{admonition} Example: how the boundaries give a score probability
:class: tip ds-example-card

For illustration, suppose $\eta_i=0$, $c_5=0$ and $c_6=1$. Then the probability of score 6 is $\sigma(1)-\sigma(0)\approx0.231$. The boundaries do not deterministically turn the predictor into a score; they determine the probabilities of the possible scores.
:::

#### Where do the cutpoints come from?

They are unknown parameters estimated jointly with the regression coefficients. The appendix specifies a prior for the first boundary and for positive increments:

::::{div}
:class: equation-sequence

$$
\begin{aligned}
c_1&\sim\mathcal N(-4,0.2^2),\\
\log d_j&\sim\mathcal N(-0.5,0.3^2),\\
c_j&=c_{j-1}+d_j+0.3.
\end{aligned}
$$
::::

Because $d_j$ is positive, the construction keeps the boundaries ordered and at least 0.3 apart. Those prior settings are inputs to fitting, not the fitted cutpoints. The observed ratings then determine which combinations of coefficients and boundaries are plausible under these assumptions; the posterior describes their joint uncertainty. The priors also anchor the latent location: shifting every cutpoint and the intercept by the same amount otherwise leaves the score probabilities unchanged.

The paper's scale notation is inconsistent: the main text says 1–10, while the appendix table and current example code use ten boundaries for 0–10 ratings. The principle is always **$K$ categories require $K-1$ finite cutpoints**.

| Component | Specification |
| --- | --- |
| Likelihood | $Y_i\mid\beta,c\sim\operatorname{OrderedLogistic}(\eta_i,c)$, with ordered category thresholds $c$. |
| Prior | $\beta_0,\beta_1,\beta_2\sim\mathcal N(0,1)$. The appendix anchors $c_1\sim\mathcal N(-4,0.2^2)$ and constructs later thresholds using positive LogNormal increments plus 0.3. |
| Posterior target | $p(\beta_0,\beta_1,\beta_2,c\mid D)$; estimate how much the autograder differs from the human and how much LLM A differs from B, with uncertainty; the cutpoints are estimated jointly. |

With the paper's $-1,+1$ coding, the contrast between two levels is $2\beta$, not $\beta$. Positive coefficients move probability toward higher score categories; they do not add a fixed number of rating points.

```{figure} figures/judge-biases/figure-1.png
:alt: Simulated score distributions for two LLMs and two graders beside posterior grader and LLM effects with 95 percent credible intervals.

Figure 1. The left panel compares simulated scores across LLMs and graders. The right panel shows posterior effects with 95% credible intervals; zero means no effect. The autograder minus human effect is negative and A minus B is positive, with both intervals excluding zero. The autograder is stricter while A receives higher scores after accounting for grader identity.
```

## Preference for its own model

#### Ordered logistic regression with interactions

A grader main effect only captures general strictness. Adding a **grader × LLM interaction** allows a judge to favor a particular model beyond that model's average score. The outcome is still an ordered rating, so the likelihood remains ordered logistic.

$$
\eta_i=\beta_0+a_{g_i}+b\,x_i^{\mathrm{LLM}}+h_{g_i,\ell_i}.
$$

Here $a_g$ is grader strictness, $b$ is the overall LLM effect, and $h_{g,\ell}$ is the additional effect of a specific grader–LLM combination. For example, $h_{A,A}-h_{A,B}$ describes how differently grader A treats the two models beyond the overall LLM effect. The same cutpoint mechanism maps the resulting predictor to score probabilities; $c$ is estimated for this fitted model rather than copied from the previous example.

| Component | Specification |
| --- | --- |
| Likelihood | $Y_i\mid\theta\sim\operatorname{OrderedLogistic}(\eta_i,c)$. |
| Prior | Normal $\mathcal N(0,1)$ priors on the intercept and free main effect and interaction coefficients; the same threshold priors as the grader strictness model. |
| Posterior target | $p(\beta_0,a,b,h,c\mid D)$; estimate whether a particular grader favors A over B beyond the overall LLM difference, and compare that preference across graders. |

```{figure} figures/judge-biases/figure-2.png
:alt: Simulated human and autograder scores beside main effect and grader and LLM interaction posterior intervals.

Figure 2. The left panel compares score distributions. The right panel adds interaction rows to the main effects; zero remains the no effect reference. Autograder A's interaction pattern suggests a preference for A over B, but the relevant displayed intervals overlap zero. The paper treats the suggestion as uncertain, not definitive evidence of self bias. The overall A advantage and lower autograder scores remain distinct effects.
```

## Grader groups

#### Hierarchical ordered logistic regression

Multiple humans and autograders differ individually, but belong to two broader groups. A hierarchy estimates both the group difference and individual variation, sharing information across judges of the same type through partial pooling. Ordered logistic still models the rating categories.

::::{div}
:class: equation-sequence

$$
\begin{aligned}
\eta_i&=\beta_0+a_{g_i}+b\,x_i^{\mathrm{LLM}},\\
a_g\mid\mu_{t(g)},\tau_{t(g)}&\sim\mathcal N(\mu_{t(g)},\tau_{t(g)}^2).
\end{aligned}
$$
::::

The group label $t(g)$ is human or autograder. $\mu_t$ is its average scoring tendency and $\tau_t$ its between judge spread. Both are estimated from the data alongside individual effects. A judge with few ratings is pulled more toward its group tendency, while a well observed judge can differ when the data support it. The score cutpoints are still estimated jointly; the hierarchy changes the prior on grader effects, not the type of observed outcome.

| Component | Specification |
| --- | --- |
| Likelihood | $Y_i\mid\theta\sim\operatorname{OrderedLogistic}(\eta_i,c)$. |
| Prior | The conditional normal distribution above is the **prior for grader effects**. Hyperpriors: $\mu_t\sim\mathcal N(0,3^2)$ and $\tau_t\sim\operatorname{HalfCauchy}(1)$. Baseline, LLM and threshold priors remain as above. |
| Posterior target | $p(\beta_0,a,b,\mu,\tau,c\mid D)$; examine $\mu_{\mathrm{human}}-\mu_{\mathrm{auto}}$, individual effects and group spreads. |

```{figure} figures/judge-biases/figure-3.png
:alt: Scores from three humans and three autograders beside posterior group means and individual grader effects.

Figure 3. The left panel shows variation among individual humans and autograders. The right panel summarizes group means and individual grader effects with 95% credible intervals. Humans generally score higher, while Autograder C appears closer to human scoring. This indicates alignment in scoring behavior, not correctness against ground truth.
```

## Item effects and agreement

#### Ordered logistic regression with item interactions

Repeated answers to the same questions allow the model to distinguish generally difficult items from questions on which particular judges disagree. Add item effects $u_m$ and grader × item effects $v_{g,m}$; keep the ordinal likelihood because the outcome remains a rating. Here $u_m$ is a shared across the question shift shared across graders, while $v_{g,m}$ captures an additional shift for one grader on that question. These effects and the cutpoints are learned together from repeated ratings.

$$
\eta_i=\beta_0+a_{g_i}+b\,x_i^{\mathrm{LLM}}+u_{m_i}+v_{g_i,m_i}.
$$

| Component | Specification |
| --- | --- |
| Likelihood | $Y_i\mid\theta\sim\operatorname{OrderedLogistic}(\eta_i,c)$. |
| Prior | Normal priors on free main effects and interactions, plus the threshold priors. The code treats item as a main effect; the appendix does not separately list an item prior bullet. |
| Posterior target | $p(\beta_0,a,b,u,v,c\mid D)$; estimate which questions systematically receive higher or lower ratings and whether any judge treats a particular question differently. |

```{figure} figures/judge-biases/figure-4.png
:alt: Simulated score distributions by question, LLM and grader beside item and grader and item posterior effects.

Figure 4. The left panel separates score distributions by item, LLM and grader. The right panel shows item main effects and grader–item interactions. Item 1 receives higher scores and item 4 lower scores. The interaction intervals are small and uncertain, giving no clear evidence that grader disagreement is concentrated on particular items.
```

The authors then simulate ratings from posterior draws and compute Krippendorff's $\alpha$, an agreement measure rather than accuracy. They repeat this after setting grader main effects to zero, asking what agreement would look like without systematic strictness differences.

$$
\text{Posterior draws}
\;\longrightarrow\;\text{Simulated ratings}
\;\longrightarrow\;\alpha\text{ for each simulated dataset}.
$$

```{figure} figures/judge-biases/figure-5.png
:alt: Observed alpha labeled minus 0.042, fitted predictive mean minus 0.056, and bias adjusted predictive mean 0.647.

Figure 5. The observed marker and fitted model distribution indicate low agreement. Removing grader main effects shifts simulated agreement upward. The plotted values are observed $\alpha=-0.042$, fitted predictive mean −0.056 and adjusted mean 0.647; the narrative instead summarizes the change as approximately −0.2 to 0.7. The improvement is a model based counterfactual, not a new set of judgments.
```

## Length preference

#### Hierarchical binary logistic regression

Judges now choose between two answers, so the outcome is binary rather than an ordered rating. A Bernoulli likelihood replaces Ordered Logistic. Pair effects capture preferences between models; a separate length slope for each grader allow humans and autograders to respond differently to answer length.

::::{div}
:class: equation-sequence

$$
\begin{aligned}
\eta_i&=\beta_0+q_{\pi_i}+a_{g_i}+s_{g_i}x_i^{\mathrm{lengthDiff}},\\
p_i&=\operatorname{logit}^{-1}(\eta_i).
\end{aligned}
$$
::::

Here $\pi_i$ identifies the LLM pair, $q_{\pi_i}$ is its fitted preference effect, and $y_i=1$ means the first listed answer wins. A positive length difference means that answer is longer. The slope $s_g$ describes how much that difference changes the log odds of selection for grader $g$; $a_g$ is its baseline shift toward the first listed answer. Unlike the ordinal models, this model needs no $c$: the inverse logit function directly turns $\eta_i$ into the probability of a binary choice.

| Component | Specification |
| --- | --- |
| Likelihood | $y_i\mid\theta\sim\operatorname{Bernoulli}(p_i)$, equivalently $\operatorname{Binomial}(1,p_i)$. No cutpoints. |
| Prior | Normal priors on baseline and free categorical effects; $s_g\sim\mathcal N(\mu_\ell,\tau_\ell^2)$, $\mu_\ell\sim\mathcal N(0,0.5^2)$, $\tau_\ell\sim\operatorname{HalfNormal}(1)$. |
| Posterior target | $p(\beta_0,q,a,s,\mu_\ell,\tau_\ell\mid D)$; estimate how strongly each grader responds to length, how graders differ, and which answer is preferred within each LLM pair. |

```{figure} figures/judge-biases/figure-6.png
:alt: Pairwise preference rates for three model pairs and posterior pair, grader and length slope effects.

Figure 6. The left panel shows observed pairwise preference rates with 95% confidence intervals. The right panel shows posterior effects with 95% credible intervals. Length sensitivity is positive, particularly for the autograder, and preferences follow A > B > C. The model can represent cyclic preferences, but this simulation does not show a cycle.
```

The current Q5 code standardizes length difference, so its slope is per standard deviation increase, not per token. In observational data, a length association does not by itself establish a causal preference for verbosity independent of unmeasured quality.

## MT-Bench results

#### Pairwise models with question, grader, length and position effects

Question effects account for repeated items; grader effects and bias terms separate model preferences from who judged the answer, its length and its displayed position. The authors select M3 as the best balance of fit and complexity among seven specifications.

| Model | Specification |
| --- | --- |
| M1 / M2 | Question effects only / grader effects. |
| M3 | Question and grader effects, plus position and length biases. |
| M4 / M5 | M3 plus quadratic length / grader × model × position interaction. |
| M6 / M7 | M3 with tighter priors / a without a hierarchy version of M3. |

```{figure} figures/judge-biases/figure-7.png
:alt: MT-Bench model comparison and posterior main effects, including length sensitivity for each grader and model shifts.

Figure 7. The left panel compares MT-Bench model specifications using predictive performance estimated with WAIC; the authors select M3 as balancing fit and complexity. The right panel shows estimated effects with uncertainty. GPT-4 has a stronger estimated length association and shifts that vary with the generating model, while humans show mild position bias. The Claude shift and human length effect intervals include zero.
```

The table reports fitted regression coefficients followed by uncertainty intervals. For example, **1.09 [0.68, 1.51]** is the estimated GPT-4 length coefficient and its reported interval, not an increase of 1.09 rating points. A positive length coefficient means that a longer first listed answer is more likely to be selected, holding the modeled predictors fixed. An interval crossing zero leaves the direction uncertain. The reported “shifts” are fitted preference effects associated with which LLM produced the answer.

| Reported effect | Estimate [reported interval] |
| --- | --- |
| GPT-4 length effect | **1.09 [0.68, 1.51]** |
| Human length effect | **0.25 [−0.02, 0.53]** |
| GPT-4 grader shift for GPT-4 outputs | **2.20 [0.29, 4.02]** |
| GPT-4 grader shift for Claude outputs | **1.56 [−0.16, 3.46]** |
| GPT-4 grader shift for Llama outputs | **−2.91 [−4.97, −0.86]** |
| Human shifts for GPT-4 / Claude / Llama | **0.34 / 0.41 / −0.75**; intervals not given in the narrative. |
| Human position effect | **0.34 [0.08, 0.61]** |

These are regression effects, not percentage point changes. The paper reports recovering the original MT-Bench ranking and bias types from the released subset, but does not provide all case study preprocessing and fitting details in the main text.

::::::::::::
